In [1]:
# imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import random

In [2]:
class_df = pd.read_csv("E:/Datasets/AIA-image-classification/images.csv")
folder = "E:/Datasets/AIA-image-classification/images"
class_df.head(5)

,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4


In [3]:
clean_names = []
clean_paths = []
clean_labels = []
missing_files = []

for index, row in class_df.iterrows():
    image_id = str(row['image_name'])
    label = row['label']
    
    full_path = os.path.join(folder, image_id)
    
    if os.path.exists(full_path):
        clean_names.append(image_id)
        clean_paths.append(full_path)
        clean_labels.append(label)
    else:
        missing_files.append(image_id)

# ✅ Create dataframe (ALL columns aligned)
df = pd.DataFrame({
    'image_name': clean_names,
    'image_paths': clean_paths,
    'label': clean_labels
})

# Required for generator
df['label'] = df['label'].astype(str)

print("Missing files:", missing_files)
print(df.info())
print(df['label'].value_counts())
df.head(10)

Missing files: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14034 entries, 0 to 14033
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   image_name   14034 non-null  object
 1   image_paths  14034 non-null  object
 2   label        14034 non-null  object
dtypes: object(3)
memory usage: 329.0+ KB
None
label
3    2512
2    2404
5    2382
4    2274
1    2271
0    2191
Name: count, dtype: int64


,image_name,image_paths,label
0,0.jpg,E:/Datasets/AIA-image-classification/images\0.jpg,0
1,1.jpg,E:/Datasets/AIA-image-classification/images\1.jpg,4
2,2.jpg,E:/Datasets/AIA-image-classification/images\2.jpg,5
3,4.jpg,E:/Datasets/AIA-image-classification/images\4.jpg,0
4,7.jpg,E:/Datasets/AIA-image-classification/images\7.jpg,4
5,8.jpg,E:/Datasets/AIA-image-classification/images\8.jpg,1
6,9.jpg,E:/Datasets/AIA-image-classification/images\9.jpg,5
7,10.jpg,E:/Datasets/AIA-image-classification/images\10...,2
8,12.jpg,E:/Datasets/AIA-image-classification/images\12...,5
9,13.jpg,E:/Datasets/AIA-image-classification/images\13...,2


In [4]:
from PIL import Image

bad_images = []
for image in df['image_paths']:
    try:
        img = Image.open(image)  
        img.verify()             
    except:
        bad_images.append(image)

len(bad_images)

0

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_generator = ImageDataGenerator(rescale=1./255)
val_generator = ImageDataGenerator(rescale=1./255)

In [6]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,      # 20% validation
    random_state=42,
    stratify=df['label']   # keeps class balance
)

In [7]:
train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),   # 🔥 smaller = faster + easier
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_iterator = val_generator.flow_from_dataframe(
    val_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 11227 validated image filenames belonging to 6 classes.
Found 2807 validated image filenames belonging to 6 classes.


In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    
    # Conv Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    # Conv Block 3
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),


    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu')
        
    layers.Dense(6, activation='softmax')
])

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_iterator,
    validation_data=val_iterator,
    epochs=15
)

Epoch 1/15
317/351 [==========================>...] - ETA: 5s - loss: 1.1859 - accuracy: 0.5001